In [49]:
# ==========================================
# Sentiment Analysis of Self-Assessed Student Competence
# Using Random Forest Algorithm
# ==========================================

# 1. Import Libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample

# Download necessary NLTK data (only first run)
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("universal_tagset")

# ==========================================
# 2. Load Dataset
# ==========================================
data = pd.read_csv("datas_labeled.csv")
data.columns = ["Student Sentiment", "Classification"]

# ==========================================
# 3. Text Preprocessing
# ==========================================
def text_preprocessing(text):
    lemmatizer = WordNetLemmatizer()
    # keep negations in stopwords
    stop_words = set(stopwords.words("english")) - {"no", "not", "don", "don’t"}

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    word_tag_tuples = pos_tag(tokens, tagset="universal")

    tag_dict = {"NOUN": "n", "VERB": "v", "ADJ": "a", "ADV": "r"}
    final_tokens = []
    for word, tag in word_tag_tuples:
        if word not in stop_words and len(word) > 1:
            if tag in tag_dict:
                final_tokens.append(lemmatizer.lemmatize(word, tag_dict[tag]))
            else:
                final_tokens.append(lemmatizer.lemmatize(word))

    return " ".join(final_tokens)

# ==========================================
# 4. Data Cleaning
# ==========================================
data.dropna(inplace=True)
data = data.drop_duplicates(subset=["Student Sentiment"])
data["Student Sentiment"] = data["Student Sentiment"].apply(text_preprocessing)

# Encode sentiment labels
data["Classification"] = data["Classification"].map({"Weak": 0, "Normal": 1, "Strong": 2})

# ==========================================
# 5. Balance the Dataset
# ==========================================
df_strong = data[data.Classification == 2]
df_normal = data[data.Classification == 1]
df_weak = data[data.Classification == 0]

# Upsample minority classes
max_count = max(len(df_strong), len(df_normal), len(df_weak))
df_strong_up = resample(df_strong, replace=True, n_samples=max_count, random_state=42)
df_normal_up = resample(df_normal, replace=True, n_samples=max_count, random_state=42)
df_weak_up = resample(df_weak, replace=True, n_samples=max_count, random_state=42)

data_balanced = pd.concat([df_strong_up, df_normal_up, df_weak_up])

print("\n✅ Data balanced successfully!")
print(data_balanced["Classification"].value_counts())

# ==========================================
# 6. TF-IDF Feature Extraction
# ==========================================
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 3))
X = vectorizer.fit_transform(data_balanced["Student Sentiment"])
y = data_balanced["Classification"]

# ==========================================
# 7. Split Data
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# 8. Random Forest Model
# ==========================================
rf_model = RandomForestClassifier(
    n_estimators=1500,        # more trees for higher accuracy
    max_depth=50,             # prevent overfitting
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="log2",      # random subset of features
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# ==========================================
# 9. Evaluation
# ==========================================
y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\n🎯 Improved Model Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ==========================================
# End of Program
# ==========================================


[nltk_data] Downloading package punkt to C:\Users\Sherylle
[nltk_data]     Rose\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Sherylle
[nltk_data]     Rose\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Sherylle
[nltk_data]     Rose\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Sherylle Rose\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to C:\Users\Sherylle
[nltk_data]     Rose\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!



✅ Data balanced successfully!
Classification
2    495
1    495
0    495
Name: count, dtype: int64

🎯 Improved Model Accuracy: 80.47%

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.87      0.79        99
           1       0.88      0.73      0.80        99
           2       0.85      0.82      0.84        99

    accuracy                           0.80       297
   macro avg       0.82      0.80      0.81       297
weighted avg       0.82      0.80      0.81       297


Confusion Matrix:
 [[86  5  8]
 [21 72  6]
 [13  5 81]]
